# Tiny LLM Decoder 7 (Generation Decoding - Sampling)

In the seventh version we are focusing on inference using sampling.

In [285]:
import numpy as np
import scipy.stats as stats
import torch
import torch.nn as nn
from torch.nn import functional as F

In [286]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## LLM Architecture

At high level, the training process is as follows:

```text

Text_processing -> Transformer Network -> Compare Loss -> Back Propagation
```

The inference process is as follows:

```text
Text_processing -> Transformer Network -> Predict Next Token -> Repeat until token size reached
```

### Corpus

In [287]:
# Text corpus
corpus = """
mary had a little lamb
its fleece was white as snow
and everywhere that mary went
the lamb was sure to go
mary had a little lamb
the lamb was sure to go
"""


### Parameters

In [288]:
# Set Parameters

# random seed
random_seed = 42
torch.manual_seed(random_seed)

# d_model is embedding dimension
d_model = 4

# block_size is the same as context window or maximum sequence length
block_size = 5



## Text Processing and Tokenization

In [289]:
# Tokenization

def tiny_tokenization(corpus_or_prompt):
  return corpus_or_prompt.split()

# Split corpus into words
word_tokens = tiny_tokenization(corpus)

# Use set to isolate unique words
vocab = sorted(list(set(word_tokens)))
vocab_size = len(vocab)

# Create token id and create lookup table from word to token id
# i is the index or tken id in this case, so we assign token id using sequential number
words_to_id_ref = {word: i for i, word in enumerate(vocab)}

# Create lookup table from id to words
id_to_words_ref = {i: word for i, word in enumerate(vocab)}

In [290]:
vocab_size

19

In [291]:
# convert corpus to id
corpus_in_id = [words_to_id_ref[word] for word in word_tokens]


## Class and Function Setup

In [292]:
def get_sequence_block(block_size, start_position, corpus_in_id):
  """
  This function extract the block in id from the corpus in id
  User need to supply the block size and the start position
  Return: X, Y, next_start_position
  """
  X = corpus_in_id[start_position: start_position+block_size]
  Y = corpus_in_id[start_position+1: start_position+block_size+1]
  return X, Y

In [293]:
def get_N(corpus_in_id, block_size):
  """
  This function calculate the number of block we need to run for the entire corpus
  """
  N = (len(corpus_in_id)-1) // block_size
  return N

In [294]:
class TinyGPT(nn.Module):

    def __init__(self, vocab_size, d_model):
        super().__init__()

        # Initialize trainable components here
        self.embeddings = nn.Embedding(vocab_size, d_model)
        self.WQ = nn.Parameter(torch.randn(d_model, d_model))
        self.WK = nn.Parameter(torch.randn(d_model, d_model))
        self.WV = nn.Parameter(torch.randn(d_model, d_model))

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model, bias=False),
            nn.ReLU(),
            nn.Linear(d_model, d_model, bias=False)
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, token_ids):

        # Perform Transformer calculations here
        # Word embeddings + positional encoding
        X_input = self.embeddings(token_ids)
        position = torch.arange(
            len(token_ids),
            dtype=X_input.dtype,
            device=X_input.device
        ).reshape(-1, 1)
        X = X_input + position

        # Q, K, V projections
        Q = X @ self.WQ
        K = X @ self.WK
        V = X @ self.WV

        # Scaled dot-product attention
        attention_score = Q @ K.T
        stable_factor = self.WQ.shape[-1] ** 0.5
        attention_stable = attention_score / stable_factor

        # Causal masking + softmax
        causal_mask = torch.triu(
            torch.full_like(attention_stable, float("-inf")),
            diagonal=1
        )
        attention_stable_mask = attention_stable + causal_mask
        attention_softmax = torch.softmax(attention_stable_mask, dim=-1)

        # Attention output + residual + norm1
        attention_output = attention_softmax @ V

        # Residual Layer
        attention_output_residual = attention_output + X

        # norm1 layer
        attention_output_residual_scaled = self.norm1(attention_output_residual)

        # FFN + residual + norm2
        ffn_1 = self.ffn(attention_output_residual_scaled)

        # Residual 2 Layer
        seond_residual = ffn_1 + attention_output_residual_scaled

        # Norm 2 Layer
        transformer_output = self.norm2(seond_residual)

        # Vocabulary projection
        logits = transformer_output @ self.embeddings.weight.T

        return logits

### Training Setup

In [295]:
def llm_train(corpus_in_id, block_size, model, criterion, optimizer, epoch=10):

  # Get N
  N = get_N(corpus_in_id, block_size=block_size)

  for i in range(epoch):

    print(f"epoch {i}")

    for j in range(N):

      # Get block
      X_id, Y_id = get_sequence_block(
          block_size=block_size,
          start_position=j * block_size,
          corpus_in_id=corpus_in_id
      )
      X_tensor = torch.tensor(X_id, dtype=torch.long, device=device)
      Y_tensor = torch.tensor(Y_id, dtype=torch.long, device=device)

      optimizer.zero_grad()

      logits = model(X_tensor)
      loss = criterion(logits, Y_tensor)
      print(f"Block {j} : Loss = {loss.item():.4f}")

      loss.backward()
      optimizer.step()

### Train the Entire Corpus

In [296]:
model = TinyGPT(vocab_size=vocab_size, d_model=d_model).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [297]:
llm_train(
    corpus_in_id=corpus_in_id,
    block_size=block_size,
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    epoch=5000
)

Streaming output truncated to the last 5000 lines.
Block 4 : Loss = 0.0118
Block 5 : Loss = 0.0075
epoch 4286
Block 0 : Loss = 0.0131
Block 1 : Loss = 0.0214
Block 2 : Loss = 0.0216
Block 3 : Loss = 0.0275
Block 4 : Loss = 0.0118
Block 5 : Loss = 0.0075
epoch 4287
Block 0 : Loss = 0.0131
Block 1 : Loss = 0.0214
Block 2 : Loss = 0.0215
Block 3 : Loss = 0.0274
Block 4 : Loss = 0.0118
Block 5 : Loss = 0.0075
epoch 4288
Block 0 : Loss = 0.0131
Block 1 : Loss = 0.0214
Block 2 : Loss = 0.0215
Block 3 : Loss = 0.0273
Block 4 : Loss = 0.0118
Block 5 : Loss = 0.0075
epoch 4289
Block 0 : Loss = 0.0131
Block 1 : Loss = 0.0213
Block 2 : Loss = 0.0215
Block 3 : Loss = 0.0273
Block 4 : Loss = 0.0118
Block 5 : Loss = 0.0075
epoch 4290
Block 0 : Loss = 0.0131
Block 1 : Loss = 0.0213
Block 2 : Loss = 0.0215
Block 3 : Loss = 0.0272
Block 4 : Loss = 0.0118
Block 5 : Loss = 0.0075
epoch 4291
Block 0 : Loss = 0.0131
Block 1 : Loss = 0.0213
Block 2 : Loss = 0.0215
Block 3 : Loss = 0.0272
Block 4 : Loss = 0.

We intentionally increase the epoch to get 100% accuracy. This is for model evaluation and inference later.

In [298]:
def evaluate_blocks(model, criterion, corpus_in_id, block_size, N):
    model.eval()

    with torch.no_grad():
        for i in range(N):
            start = i * block_size

            X_id, Y_id = get_sequence_block(
                block_size=block_size,
                start_position=start,
                corpus_in_id=corpus_in_id
            )

            X = torch.tensor(X_id, dtype=torch.long, device=device)
            Y = torch.tensor(Y_id, dtype=torch.long, device=device)

            logits = model(X)
            loss = criterion(logits, Y)

            print(f"Block {i}: Loss = {loss.item():.4f}")

            predicted_ids = torch.argmax(logits, dim=-1)
            correct = (predicted_ids == Y).sum().item()
            accuracy = correct / len(Y)

            print(f"Block {i}: Accuracy = {accuracy:.4f}")

            print("-" * 50)
            print(f"Block {i}")
            for k in range(5):
              print(f"Position {k}, expected = {id_to_words_ref[int(Y[k])]}, predicted = {id_to_words_ref[int(predicted_ids[k])]}")
            print("-" * 50)

In [299]:
N = get_N(corpus_in_id, block_size=block_size)


In [300]:
evaluate_blocks(model=model, criterion=criterion, corpus_in_id=corpus_in_id, block_size=block_size, N=N)

Block 0: Loss = 0.0086
Block 0: Accuracy = 1.0000
--------------------------------------------------
Block 0
Position 0, expected = had, predicted = had
Position 1, expected = a, predicted = a
Position 2, expected = little, predicted = little
Position 3, expected = lamb, predicted = lamb
Position 4, expected = its, predicted = its
--------------------------------------------------
Block 1: Loss = 0.0149
Block 1: Accuracy = 1.0000
--------------------------------------------------
Block 1
Position 0, expected = fleece, predicted = fleece
Position 1, expected = was, predicted = was
Position 2, expected = white, predicted = white
Position 3, expected = as, predicted = as
Position 4, expected = snow, predicted = snow
--------------------------------------------------
Block 2: Loss = 0.0148
Block 2: Accuracy = 1.0000
--------------------------------------------------
Block 2
Position 0, expected = and, predicted = and
Position 1, expected = everywhere, predicted = everywhere
Position 2, exp

## Inference Decoding - Sampling

During inference, there are many decoding method:

- Greedy
- Sampling
- Temperature Sampling
- Top K Sampling
- Top P Sampling
- Beam Search

In this exercise, we will be looking at sampling. In sampling we do not use `argmax()`, instead we use `torch.multinomail()`.

## Reference Note: LLM Decoding — Sampling vs. Greedy & PyTorch Implementation

### 1. Why Use Sampling Instead of Greedy Search (Argmax)?

In Large Language Model (LLM) decoding, **Greedy Search (Argmax)** selects the single token with the highest conditional probability at every generation step ($\text{argmax}_{w} P(w \vert{} w_{<t})$). While deterministic and optimal for exact factual tasks, it suffers from several core drawbacks in open-ended text generation:

* **Repetitiveness & Stagnation:** Greedy decoding easily gets trapped in local probability loops (e.g., repeating phrases or sentences indefinitely) because it always takes the most predictable local step.
* **Lack of Natural Diversity:** Human speech and writing rarely choose the statistically highest-probability word at every step. Greedy decoding strips away stylistic nuance, making generated text feel mechanical and overly generic.
* **No Output Variability:** Argmax is deterministic ($T = 0$). It yields the exact same output for a given prompt, preventing multi-candidate generation, creative brainstorming, or self-consistency voting.

**Sampling** introduces controlled stochasticity by sampling tokens proportional to their predicted probability distribution. To prevent sampling from selecting ungrammatical or nonsensical tokens in the "long tail" of low probabilities, it is combined with:
* **Temperature Scaling ($T$):** Adjusts the peakiness of the probability distribution before applying softmax.
* **Top-$k$ Sampling:** Limits candidate tokens to the $k$ most probable tokens.
* **Top-$p$ (Nucleus) Sampling:** Limits candidate tokens to the smallest set whose cumulative probability mass exceeds $p$.

### 2. Low-Level Core Implementation in PyTorch (`torch.multinomial`)

In PyTorch, while `torch.argmax` returns the single index with the highest probability, **`torch.multinomial`** draws samples from a multinomial probability distribution.

#### Mathematical Pipeline
1. **Raw Logits** $\rightarrow$ 2. **Temperature Scaling** ($\frac{z}{T}$) $\rightarrow$ 3. **Softmax** $\rightarrow$ 4. **`torch.multinomial`**

#### Python Example Code

In [301]:
# Example: Unnormalized logits for 5 vocabulary tokens from an LLM forward pass
logits = torch.tensor([2.0, 1.5, 0.1, -1.0, -3.0])

# 1. Apply Temperature Scaling
# Lower T (< 1.0) sharpens the distribution; Higher T (> 1.0) flattens it.
temperature = 0.8
scaled_logits = logits / temperature

# 2. Convert Logits to Probabilities using Softmax
probs = F.softmax(scaled_logits, dim=-1)
# Output probabilities: tensor([0.5610, 0.3010, 0.0805, 0.0205, 0.0169])

# --- GREEDY APPROACH (Argmax) ---
# Always selects index 0 (highest value: 0.5610)
greedy_token_id = torch.argmax(probs)

# --- SAMPLING APPROACH (Multinomial) ---
# Samples 1 token index based on the probability distribution weights.
# Index 0 has a 56.1% chance, Index 1 has a 30.1% chance, etc.
sampled_token_id = torch.multinomial(probs, num_samples=1)

### Summary Comparison Table

| Metric / Feature | `torch.argmax` (Greedy) | `torch.multinomial` (Sampling) |
| --- | --- | --- |
| **Output Type** | Deterministic | Stochastic / Probabilistic |
| **Diversity** | Low (Repetitive) | High (Natural & Creative) |
| **Primary Use Cases** | Math, Code, Fact Extraction | Dialogue, Storytelling, Summarization |
| **PyTorch Function** | `torch.argmax(probs)` | `torch.multinomial(probs, num_samples=1)` |


### Functions for Inference

In [302]:
def prompt_processing(user_prompt_text):
  user_prompt_text = user_prompt_text.split()
  prompt_id = [words_to_id_ref[each] for each in user_prompt_text]
  return torch.tensor(prompt_id, dtype=torch.long, device=device)

### Inference 1 - Greedy Decoding

Lets buils function that auto process prompts.

In [303]:
def prompt_inference_greedy(user_prompt, max_token, model, id_to_words_dict):
  X = prompt_processing(user_prompt)
  for i in range(max_token):
    model.eval()

    # the code below prevent torch for tracking the gradient since we are inferencing and not training
    with torch.no_grad():
        prompt_logits = model(X)

    # predicting token after
    token = [id_to_words_dict[each.item()] for each in X]
    print(f"Predicting: last token of the prompt {token}")

    # get the logits of last token
    predict_logits = prompt_logits[-1]

    # get the last token prediction
    next_token_probs = torch.softmax(predict_logits, dim=-1)

    # display top 5 softmax
    sorted_logits, sorted_indices = torch.sort(next_token_probs, descending=True)
    sorted_token = [id_to_words_dict[each.item()] for each in sorted_indices]
    sorted_pairs = zip(sorted_token[0:5], sorted_logits[0:5])
    for word, prob in sorted_pairs:
      print(f"{word}, prob={prob.item():.8f}")

    # next token prediction - greedy
    next_token_id = torch.argmax(next_token_probs)

    X = torch.cat([X, torch.tensor([next_token_id], dtype=torch.long, device=device)])
  return X

In [304]:
def generate_response(X, id_to_words_dict):
  response = [id_to_words_dict[each.item()] for each in X]
  print(f"Prompt: {user_prompt}")
  print(f"Response: {' '.join(response)}")

In [305]:
user_prompt = "mary"
user_prompt

'mary'

In [306]:
max_token = 4

In [307]:
X = prompt_inference_greedy(user_prompt, max_token, model, id_to_words_ref)

Predicting: last token of the prompt ['mary']
had, prob=0.98996311
go, prob=0.00642766
snow, prob=0.00309435
a, prob=0.00047261
and, prob=0.00004190
Predicting: last token of the prompt ['mary', 'had']
a, prob=0.99224997
and, prob=0.00517981
snow, prob=0.00256901
little, prob=0.00000072
go, prob=0.00000041
Predicting: last token of the prompt ['mary', 'had', 'a']
little, prob=0.99456775
and, prob=0.00269514
everywhere, prob=0.00194750
that, prob=0.00076331
lamb, prob=0.00002554
Predicting: last token of the prompt ['mary', 'had', 'a', 'little']
lamb, prob=0.99308503
everywhere, prob=0.00495472
the, prob=0.00144541
to, prob=0.00051260
little, prob=0.00000222


In [308]:
X

tensor([10,  6,  0,  9,  8], device='cuda:0')

In [309]:
generate_response(X, id_to_words_ref)

Prompt: mary
Response: mary had a little lamb


As you can see even if we set the temperature at 5000, the softmax for each token is nearly the same but the correct token probability is still higher then the rest. We still select the highest probability. In the next inference, we will use `torch.multinomial()` this function will select base on the probability. If probability is 0.7 then there is 70% chance that the token will be selected

### Inference 2 - Sampling

In [310]:
def prompt_inference_sampling(user_prompt, max_token, model, id_to_words_dict):
  X = prompt_processing(user_prompt)
  for i in range(max_token):
    model.eval()

    # the code below prevent torch for tracking the gradient since we are inferencing and not training
    with torch.no_grad():
        prompt_logits = model(X)

    # predicting token after
    token = [id_to_words_dict[each.item()] for each in X]
    print(f"Predicting: last token of the prompt {token}")

    # get the logits of last token
    predict_logits = prompt_logits[-1]

    # get the last token prediction
    next_token_probs = torch.softmax(predict_logits, dim=-1)

    # display top 5 softmax
    sorted_logits, sorted_indices = torch.sort(next_token_probs, descending=True)
    sorted_token = [id_to_words_dict[each.item()] for each in sorted_indices]
    sorted_pairs = zip(sorted_token[0:5], sorted_logits[0:5])
    for word, prob in sorted_pairs:
      print(f"{word}, prob={prob.item():.8f}")

    # next token prediction - greedy
    next_token_id = torch.multinomial(next_token_probs, num_samples=1)

    X = torch.cat([X, torch.tensor([next_token_id], dtype=torch.long, device=device)])
  return X

In [311]:
user_prompt = "mary"
user_prompt

'mary'

In [312]:
max_token = 4

In [313]:
X = prompt_inference_sampling(user_prompt, max_token, model, id_to_words_ref)

Predicting: last token of the prompt ['mary']
had, prob=0.98996311
go, prob=0.00642766
snow, prob=0.00309435
a, prob=0.00047261
and, prob=0.00004190
Predicting: last token of the prompt ['mary', 'had']
a, prob=0.99224997
and, prob=0.00517981
snow, prob=0.00256901
little, prob=0.00000072
go, prob=0.00000041
Predicting: last token of the prompt ['mary', 'had', 'a']
little, prob=0.99456775
and, prob=0.00269514
everywhere, prob=0.00194750
that, prob=0.00076331
lamb, prob=0.00002554
Predicting: last token of the prompt ['mary', 'had', 'a', 'little']
lamb, prob=0.99308503
everywhere, prob=0.00495472
the, prob=0.00144541
to, prob=0.00051260
little, prob=0.00000222


In [314]:
generate_response(X, id_to_words_ref)

Prompt: mary
Response: mary had a little lamb


We still get correct prediction as the probability of the most propable word is much higher then the second propable wprd.

### Inference 3

In [315]:
def prompt_inference_sampling_no_print(user_prompt, max_token, model, id_to_words_dict):
  X = prompt_processing(user_prompt)
  for i in range(max_token):
    model.eval()

    # the code below prevent torch for tracking the gradient since we are inferencing and not training
    with torch.no_grad():
        prompt_logits = model(X)

    # predicting token after
    token = [id_to_words_dict[each.item()] for each in X]
    #print(f"Predicting: last token of the prompt {token}")

    # get the logits of last token
    predict_logits = prompt_logits[-1]

    # get the last token prediction
    next_token_probs = torch.softmax(predict_logits, dim=-1)

    # display top 5 softmax
    sorted_logits, sorted_indices = torch.sort(next_token_probs, descending=True)
    sorted_token = [id_to_words_dict[each.item()] for each in sorted_indices]
    sorted_pairs = zip(sorted_token[0:5], sorted_logits[0:5])
    # for word, prob in sorted_pairs:
    #   print(f"{word}, prob={prob.item():.8f}")

    # next token prediction - greedy
    next_token_id = torch.multinomial(next_token_probs, num_samples=1)

    X = torch.cat([X, torch.tensor([next_token_id], dtype=torch.long, device=device)])
  return X

In [316]:
user_prompt = "mary had"
user_prompt

'mary had'

In [317]:
max_token = 3

In [318]:
X = prompt_inference_sampling_no_print(user_prompt, max_token, model, id_to_words_ref)

In [319]:
generate_response(X, id_to_words_ref)


Prompt: mary had
Response: mary had and to little


In [320]:
for i in range(100):
  X = prompt_inference_sampling_no_print(user_prompt, max_token, model, id_to_words_ref)
  generate_response(X, id_to_words_ref)

Prompt: mary had
Response: mary had a little lamb
Prompt: mary had
Response: mary had a little lamb
Prompt: mary had
Response: mary had a little lamb
Prompt: mary had
Response: mary had a little lamb
Prompt: mary had
Response: mary had a little lamb
Prompt: mary had
Response: mary had a little lamb
Prompt: mary had
Response: mary had a little lamb
Prompt: mary had
Response: mary had a little lamb
Prompt: mary had
Response: mary had a little lamb
Prompt: mary had
Response: mary had a little lamb
Prompt: mary had
Response: mary had a little lamb
Prompt: mary had
Response: mary had a little lamb
Prompt: mary had
Response: mary had a little lamb
Prompt: mary had
Response: mary had a little lamb
Prompt: mary had
Response: mary had a little lamb
Prompt: mary had
Response: mary had a little lamb
Prompt: mary had
Response: mary had a little lamb
Prompt: mary had
Response: mary had a little lamb
Prompt: mary had
Response: mary had a little lamb
Prompt: mary had
Response: mary had a little lamb


If we where to run the prompt 100 times, either 1 or 2 times, we get different response. This confirms the sampling is working. But we need make the probabilites of the possible words to be more even. Thus we introduces temperature.

## END